# Notebook 03 — Weather Data Cleaning & Munging

**Real-world data is messy.** Before we can use historical weather observations to
improve our minion-generated forecasts, we need to clean, transform, and validate
the raw data. This process is called "data munging" (or "data wrangling").

## What you will learn

1. **Loading raw data** — Read GHCN-Daily and Open-Meteo CSVs from MinIO
2. **Understanding GHCN format** — The long-format layout where each row is one measurement type (TMAX, TMIN, PRCP, etc.)
3. **Pivoting data** — Reshape from long to wide format using `pivot_table()`
4. **Handling missing values** — Detect nulls, decide between dropping and imputing
5. **Unit conversion** — GHCN stores temperatures in tenths of degrees Celsius
6. **Mapping to Summary labels** — Assign our app's labels (Freezing, Bracing, ..., Scorching) based on temperature thresholds
7. **Saving cleaned data** — Write Parquet files to MinIO and load into DuckDB

## Pre-requisites

- The `download_weather_datasets` Airflow DAG must have run at least once
- Notebook 01 is recommended (but not required) for familiarity with the data

---

### What is GHCN-Daily?

The **Global Historical Climatology Network – Daily** (GHCN-Daily) is a dataset maintained
by NOAA (the U.S. National Oceanic and Atmospheric Administration). It contains daily
weather observations from over 100,000 weather stations worldwide, some dating back
to the 1800s. Key variables include:

| Code | Meaning | Unit |
|------|---------|------|
| TMAX | Maximum temperature | Tenths of °C |
| TMIN | Minimum temperature | Tenths of °C |
| PRCP | Precipitation | Tenths of mm |
| SNOW | Snowfall | mm |
| SNWD | Snow depth | mm |

**Important:** Temperatures are stored in *tenths* of degrees Celsius. A value of
256 means 25.6°C. We must divide by 10 to get actual degrees.

In [ ]:
# Cell 1 — Imports and setup
import sys
import os
import io

SHARED_PATH = '/home/jovyan/work/shared'
if SHARED_PATH not in sys.path:
    sys.path.insert(0, SHARED_PATH)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import duckdb

from minio_helper import get_client, read_csv, object_exists, upload_dataframe

%matplotlib inline
sns.set_theme(style='whitegrid', palette='tab10')
plt.rcParams['figure.dpi'] = 120

client = get_client()
print('Imports OK')

## Part 1: Loading and Understanding GHCN-Daily Data

GHCN-Daily files use a **long format**: each row represents one measurement
for one station on one date. A single day at a single station might have
5+ rows (TMAX, TMIN, PRCP, SNOW, SNWD, etc.).

This is different from the **wide format** you might expect, where each row
is a day and each column is a variable.

In [ ]:
# Cell 3 — Load a GHCN-Daily station CSV from MinIO
# We'll start with New York Central Park (USW00094728) which has data from 1869.

BUCKET = 'weather-raw'
GHCN_OBJECT = 'ghcn/USW00094728.csv'

if not object_exists(client, BUCKET, GHCN_OBJECT):
    raise FileNotFoundError(
        f'{BUCKET}/{GHCN_OBJECT} not found in MinIO. '
        'Run the download_weather_datasets Airflow DAG first.'
    )

# GHCN CSVs have no header row. The columns are:
# STATION, DATE, ELEMENT, VALUE, M_FLAG, Q_FLAG, S_FLAG, OBS_TIME
ghcn_raw = read_csv(
    client, BUCKET, GHCN_OBJECT,
    header=None,
    names=['station', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time'],
    parse_dates=['date'],
    dtype={'station': str, 'element': str, 'value': float},
)

print(f'Shape: {ghcn_raw.shape}')
print(f'Date range: {ghcn_raw["date"].min().date()} to {ghcn_raw["date"].max().date()}')
print(f'\nUnique elements: {sorted(ghcn_raw["element"].unique())}')
print(f'\nSample rows:')
ghcn_raw.head(10)

In [ ]:
# Cell 4 — Filter to recent years and core elements
# The full dataset can span 150+ years. For our purposes, let's focus on
# the last 5 years of data and the temperature/precipitation elements.

ELEMENTS_OF_INTEREST = ['TMAX', 'TMIN', 'PRCP', 'SNOW', 'SNWD']
START_DATE = '2019-01-01'

ghcn_filtered = ghcn_raw[
    (ghcn_raw['element'].isin(ELEMENTS_OF_INTEREST)) &
    (ghcn_raw['date'] >= START_DATE)
].copy()

print(f'Filtered from {len(ghcn_raw):,} to {len(ghcn_filtered):,} rows')
print(f'Date range: {ghcn_filtered["date"].min().date()} to {ghcn_filtered["date"].max().date()}')
print(f'\nRows per element:')
ghcn_filtered['element'].value_counts()

## Part 2: Pivoting from Long to Wide Format

Right now each row is one measurement. We want each row to be one **day**,
with columns for TMAX, TMIN, PRCP, etc. This is called a **pivot**.

```
BEFORE (long):                     AFTER (wide):
date        element  value         date        TMAX  TMIN  PRCP
2024-01-01  TMAX     50            2024-01-01  50    -10   0
2024-01-01  TMIN     -10           2024-01-02  72    30    25
2024-01-01  PRCP     0
2024-01-02  TMAX     72
...
```

In [ ]:
# Cell 6 — Pivot long to wide
# pivot_table() reshapes the DataFrame: rows=date, columns=element, values=value.
# aggfunc='first' handles the rare case where a station reports the same
# element twice on the same day (takes the first value).

ghcn_wide = ghcn_filtered.pivot_table(
    index='date',
    columns='element',
    values='value',
    aggfunc='first',
)

# Reset index so 'date' becomes a regular column again
ghcn_wide = ghcn_wide.reset_index()

# Flatten the column names (pivot_table creates a MultiIndex)
ghcn_wide.columns.name = None

print(f'Wide format shape: {ghcn_wide.shape}')
print(f'Columns: {list(ghcn_wide.columns)}')
ghcn_wide.head()

## Part 3: Handling Missing Values

Missing data is inevitable in real-world datasets. Weather stations go
offline, sensors malfunction, and some elements aren't recorded at every
station. There are three common strategies:

1. **Drop rows** with missing values — Simple but loses data
2. **Fill forward/backward** — Use the previous/next day's value
3. **Interpolate** — Estimate missing values based on surrounding data

For temperature, interpolation is reasonable because temperature changes
gradually from day to day. For precipitation, filling with 0 or dropping
is better because rain is sporadic.

In [ ]:
# Cell 8 — Detect missing values
# isnull() returns True for each missing cell.
# Summing gives the total count of missing values per column.

null_counts = ghcn_wide.isnull().sum()
total_rows = len(ghcn_wide)

print('Missing values per column:')
print('=' * 40)
for col in null_counts.index:
    count = null_counts[col]
    pct = 100 * count / total_rows
    status = 'OK' if count == 0 else f'{count:,} ({pct:.1f}%)'
    print(f'  {col:10s}  {status}')

# Visualize the pattern of missing data
fig, ax = plt.subplots(figsize=(10, 3))
missing_matrix = ghcn_wide.set_index('date')[ELEMENTS_OF_INTEREST].isnull().astype(int)
ax.imshow(
    missing_matrix.T,
    aspect='auto',
    cmap='Reds',
    interpolation='nearest',
)
ax.set_yticks(range(len(ELEMENTS_OF_INTEREST)))
ax.set_yticklabels(ELEMENTS_OF_INTEREST)
ax.set_title('Missing Data Pattern (red = missing)', fontsize=12)
ax.set_xlabel('Day index')
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 — Handle missing values
# Strategy:
#   - TMAX/TMIN: linear interpolation (temperature changes smoothly day-to-day)
#   - PRCP/SNOW/SNWD: fill with 0 (a missing precipitation reading most likely
#     means "no precipitation was recorded", not that data is truly unknown)
#   - After interpolation, drop any remaining rows where TMAX or TMIN is still null
#     (these would be at the very start/end of the series where interpolation can't help)

ghcn_clean = ghcn_wide.copy()

# Interpolate temperature columns
for col in ['TMAX', 'TMIN']:
    if col in ghcn_clean.columns:
        before_nulls = ghcn_clean[col].isnull().sum()
        ghcn_clean[col] = ghcn_clean[col].interpolate(method='linear')
        after_nulls = ghcn_clean[col].isnull().sum()
        print(f'{col}: interpolated {before_nulls - after_nulls} missing values')

# Fill precipitation/snow with 0
for col in ['PRCP', 'SNOW', 'SNWD']:
    if col in ghcn_clean.columns:
        filled = ghcn_clean[col].isnull().sum()
        ghcn_clean[col] = ghcn_clean[col].fillna(0)
        if filled > 0:
            print(f'{col}: filled {filled} missing values with 0')

# Drop rows where temperature is still null (edges of the series)
before_len = len(ghcn_clean)
ghcn_clean = ghcn_clean.dropna(subset=['TMAX', 'TMIN'])
dropped = before_len - len(ghcn_clean)
if dropped > 0:
    print(f'Dropped {dropped} rows with remaining null temperatures')

print(f'\nFinal shape: {ghcn_clean.shape}')
print(f'Remaining nulls: {ghcn_clean.isnull().sum().sum()}')

## Part 4: Unit Conversion

GHCN stores temperatures in **tenths of degrees Celsius**:
- A TMAX value of `256` means **25.6°C**
- A TMAX value of `-50` means **-5.0°C**

PRCP is stored in **tenths of millimeters**:
- A PRCP value of `120` means **12.0 mm** of precipitation

We need to divide by 10 to get standard units. This is a common gotcha
with GHCN data — always check the documentation!

In [ ]:
# Cell 11 — Convert units
# Show before/after to make the conversion visible

print('BEFORE conversion (sample):')
print(ghcn_clean[['date', 'TMAX', 'TMIN', 'PRCP']].head(3).to_string(index=False))

# Divide temperature and precipitation by 10
for col in ['TMAX', 'TMIN']:
    if col in ghcn_clean.columns:
        ghcn_clean[col] = ghcn_clean[col] / 10.0

if 'PRCP' in ghcn_clean.columns:
    ghcn_clean['PRCP'] = ghcn_clean['PRCP'] / 10.0

# Compute a daily mean temperature (average of max and min)
# This is a standard approximation used in climatology.
ghcn_clean['TMEAN'] = (ghcn_clean['TMAX'] + ghcn_clean['TMIN']) / 2.0

print('\nAFTER conversion (sample):')
print(ghcn_clean[['date', 'TMAX', 'TMIN', 'TMEAN', 'PRCP']].head(3).to_string(index=False))

print('\nDescriptive statistics (converted):')
ghcn_clean[['TMAX', 'TMIN', 'TMEAN', 'PRCP']].describe().round(1)

In [ ]:
# Cell 12 — Before/after histogram comparison
# Let's visualize how the temperature distribution looks after cleaning.

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: raw TMAX from the original data (before conversion)
raw_tmax = ghcn_wide['TMAX'].dropna() / 10.0  # convert for comparison
axes[0].hist(raw_tmax, bins=50, edgecolor='white', alpha=0.7, color='steelblue')
axes[0].set_title('TMAX Distribution (before cleaning)', fontsize=11)
axes[0].set_xlabel('Temperature (°C)')
axes[0].set_ylabel('Number of days')
axes[0].axvline(raw_tmax.mean(), color='red', linestyle='--', label=f'Mean: {raw_tmax.mean():.1f}°C')
axes[0].legend()

# Right: cleaned TMAX with missing values interpolated
axes[1].hist(ghcn_clean['TMAX'], bins=50, edgecolor='white', alpha=0.7, color='coral')
axes[1].set_title('TMAX Distribution (after cleaning)', fontsize=11)
axes[1].set_xlabel('Temperature (°C)')
axes[1].set_ylabel('Number of days')
axes[1].axvline(ghcn_clean['TMAX'].mean(), color='red', linestyle='--', label=f'Mean: {ghcn_clean["TMAX"].mean():.1f}°C')
axes[1].legend()

fig.suptitle('New York Central Park — Temperature Before vs After Cleaning', fontsize=13)
plt.tight_layout()
plt.show()

## Part 5: Mapping Temperature to Summary Labels

Our weather app uses 10 summary labels. Currently, minions assign them
randomly. We want to map real temperatures to the correct labels so we
can learn the realistic distribution.

### Temperature Thresholds

These thresholds are based on common meteorological conventions:

| Label | Temperature Range | What it feels like |
|-------|------------------|--------------------|
| Freezing | < -5°C | Ice and frost; exposed skin risk |
| Bracing | -5°C to 0°C | Very cold; need heavy winter gear |
| Chilly | 0°C to 5°C | Cold; breath visible outdoors |
| Cool | 5°C to 12°C | Light jacket weather |
| Mild | 12°C to 18°C | Comfortable; light layers |
| Warm | 18°C to 24°C | T-shirt weather |
| Balmy | 24°C to 30°C | Pleasantly warm; summer day |
| Hot | 30°C to 35°C | Uncomfortable heat; stay hydrated |
| Sweltering | 35°C to 40°C | Dangerous heat; limit outdoor activity |
| Scorching | > 40°C | Extreme heat; rarely seen outside deserts |

These thresholds match the labels in `MinionSchedulerService.cs`.

In [ ]:
# Cell 14 — Map temperature to Summary labels using pd.cut()
# pd.cut() bins continuous values into discrete categories.
# The bins list defines the edges: (-inf, -5], (-5, 0], (0, 5], ...

SUMMARY_BINS = [-np.inf, -5, 0, 5, 12, 18, 24, 30, 35, 40, np.inf]
SUMMARY_LABELS = [
    'Freezing',    # < -5
    'Bracing',     # -5 to 0
    'Chilly',      # 0 to 5
    'Cool',        # 5 to 12
    'Mild',        # 12 to 18
    'Warm',        # 18 to 24
    'Balmy',       # 24 to 30
    'Hot',         # 30 to 35
    'Sweltering',  # 35 to 40
    'Scorching',   # > 40
]

# Use TMEAN (daily average) to assign the Summary label
ghcn_clean['summary'] = pd.cut(
    ghcn_clean['TMEAN'],
    bins=SUMMARY_BINS,
    labels=SUMMARY_LABELS,
)

print('Summary label distribution for New York Central Park:')
print('=' * 50)
label_counts = ghcn_clean['summary'].value_counts()
for label in SUMMARY_LABELS:
    count = label_counts.get(label, 0)
    pct = 100 * count / len(ghcn_clean)
    bar = '#' * int(pct)
    print(f'  {label:12s}  {count:5d}  ({pct:5.1f}%)  {bar}')

In [ ]:
# Cell 15 — Visualize Summary label distribution

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: Bar chart of label counts
colors = plt.cm.coolwarm(np.linspace(0, 1, len(SUMMARY_LABELS)))
label_counts_ordered = [ghcn_clean['summary'].value_counts().get(l, 0) for l in SUMMARY_LABELS]

axes[0].bar(SUMMARY_LABELS, label_counts_ordered, color=colors, edgecolor='white')
axes[0].set_title('Summary Label Distribution (New York)', fontsize=12)
axes[0].set_xlabel('Summary Label')
axes[0].set_ylabel('Number of Days')
axes[0].tick_params(axis='x', rotation=45)

# Right: Box plot of temperature by label
# This validates that the mapping is correct — each label should cover
# a distinct temperature range with no overlap.
sns.boxplot(
    data=ghcn_clean,
    x='summary',
    y='TMEAN',
    order=SUMMARY_LABELS,
    hue='summary',
    hue_order=SUMMARY_LABELS,
    palette='coolwarm',
    legend=False,
    ax=axes[1],
)
axes[1].set_title('Temperature Range by Summary Label', fontsize=12)
axes[1].set_xlabel('Summary Label')
axes[1].set_ylabel('Daily Mean Temperature (°C)')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

print('\nNotice how each label covers a distinct temperature band.')
print('This is exactly what we want — no "Scorching" days at 10°C!')

## Part 6: Process All Stations and Open-Meteo Data

Now we'll apply the same cleaning pipeline to all five GHCN stations
and the Open-Meteo data, creating a unified cleaned dataset.

In [ ]:
# Cell 17 — Process all GHCN stations

GHCN_STATIONS = [
    ('USW00094728', 'new_york'),
    ('USW00023174', 'los_angeles'),
    ('UKW00035065', 'london'),
    ('JA000047662', 'tokyo'),
    ('ASN00086282', 'melbourne'),
]

all_ghcn_clean = []

for station_id, city in GHCN_STATIONS:
    obj = f'ghcn/{station_id}.csv'
    if not object_exists(client, BUCKET, obj):
        print(f'  SKIP: {obj} not found in MinIO')
        continue

    # Load raw data
    raw = read_csv(
        client, BUCKET, obj,
        header=None,
        names=['station', 'date', 'element', 'value', 'm_flag', 'q_flag', 's_flag', 'obs_time'],
        parse_dates=['date'],
        dtype={'station': str, 'element': str, 'value': float},
    )

    # Filter to recent years and core elements
    filtered = raw[
        (raw['element'].isin(['TMAX', 'TMIN', 'PRCP'])) &
        (raw['date'] >= START_DATE)
    ]

    # Pivot to wide format
    wide = filtered.pivot_table(index='date', columns='element', values='value', aggfunc='first').reset_index()
    wide.columns.name = None

    # Interpolate temperatures, fill precipitation with 0
    for col in ['TMAX', 'TMIN']:
        if col in wide.columns:
            wide[col] = wide[col].interpolate(method='linear')
    if 'PRCP' in wide.columns:
        wide['PRCP'] = wide['PRCP'].fillna(0)

    wide = wide.dropna(subset=['TMAX', 'TMIN'])

    # Convert units
    for col in ['TMAX', 'TMIN']:
        if col in wide.columns:
            wide[col] = wide[col] / 10.0
    if 'PRCP' in wide.columns:
        wide['PRCP'] = wide['PRCP'] / 10.0

    wide['TMEAN'] = (wide['TMAX'] + wide['TMIN']) / 2.0

    # Assign Summary label
    wide['summary'] = pd.cut(wide['TMEAN'], bins=SUMMARY_BINS, labels=SUMMARY_LABELS)

    # Add metadata
    wide['station_id'] = station_id
    wide['location'] = city
    wide['source'] = 'ghcn'

    all_ghcn_clean.append(wide)
    print(f'  {city:15s}  {len(wide):5,} rows  ({wide["date"].min().date()} to {wide["date"].max().date()})')

if all_ghcn_clean:
    ghcn_all = pd.concat(all_ghcn_clean, ignore_index=True)
    print(f'\nTotal GHCN rows: {len(ghcn_all):,}')
else:
    ghcn_all = pd.DataFrame()
    print('No GHCN data loaded.')

In [ ]:
# Cell 18 — Process Open-Meteo data
# Open-Meteo data is already in wide format with temperatures in °C,
# so less cleaning is needed.

METEO_LOCATIONS = ['new_york', 'london', 'tokyo', 'melbourne', 'singapore']

all_meteo_clean = []

for loc in METEO_LOCATIONS:
    obj = f'open-meteo/{loc}.csv'
    if not object_exists(client, BUCKET, obj):
        print(f'  SKIP: {obj} not found')
        continue

    meteo = read_csv(client, BUCKET, obj, parse_dates=['date'])

    # Open-Meteo already provides temperature_2m_mean in °C
    # We just need to add the Summary label
    if 'temperature_2m_mean' in meteo.columns:
        meteo['summary'] = pd.cut(
            meteo['temperature_2m_mean'],
            bins=SUMMARY_BINS,
            labels=SUMMARY_LABELS,
        )

    meteo['source'] = 'open_meteo'
    all_meteo_clean.append(meteo)
    print(f'  {loc:15s}  {len(meteo):5,} rows')

if all_meteo_clean:
    meteo_all = pd.concat(all_meteo_clean, ignore_index=True)
    print(f'\nTotal Open-Meteo rows: {len(meteo_all):,}')
else:
    meteo_all = pd.DataFrame()
    print('No Open-Meteo data loaded.')

In [ ]:
# Cell 19 — Cross-tabulation: Summary labels by location
# pd.crosstab() creates a contingency table showing count/percentage
# of each label at each location.

if len(meteo_all) > 0:
    ct = pd.crosstab(
        meteo_all['location'],
        meteo_all['summary'],
        normalize='index',  # row percentages (each location sums to 1.0)
    ).round(3)

    # Reorder columns to match our label order
    ct = ct.reindex(columns=SUMMARY_LABELS, fill_value=0)

    print('Summary Label Distribution by Location (Open-Meteo):')
    display(ct)

    # Heatmap
    fig, ax = plt.subplots(figsize=(12, 4))
    sns.heatmap(
        ct,
        annot=True,
        fmt='.1%',
        cmap='YlOrRd',
        ax=ax,
    )
    ax.set_title('Summary Label Frequency by Location', fontsize=13)
    ax.set_xlabel('Summary Label')
    ax.set_ylabel('Location')
    plt.tight_layout()
    plt.show()

    print('\nKey observations:')
    print('  - Singapore is always Warm/Balmy/Hot (tropical climate, no winter)')
    print('  - New York and Tokyo have the widest range (distinct seasons)')
    print('  - London rarely reaches Hot or above (maritime climate)')

## Part 7: Save Cleaned Data to MinIO

We save the cleaned data in **Parquet** format rather than CSV because:
- Parquet preserves data types exactly (dates stay as dates, not strings)
- It compresses data 5-10x smaller than CSV
- It's columnar, so reading a few columns from a large file is fast
- DuckDB and pandas both read Parquet natively

In [ ]:
# Cell 21 — Save cleaned GHCN data to MinIO as Parquet

CLEAN_BUCKET = 'clean-weather'

if len(ghcn_all) > 0:
    # Select and rename columns for a consistent schema
    ghcn_export = ghcn_all[['date', 'location', 'station_id', 'source',
                            'TMAX', 'TMIN', 'TMEAN', 'PRCP', 'summary']].copy()
    ghcn_export.columns = ['date', 'location', 'station_id', 'source',
                           'temp_max_c', 'temp_min_c', 'temp_mean_c',
                           'precipitation_mm', 'summary']
    # Convert summary from Categorical to string for Parquet compatibility
    ghcn_export['summary'] = ghcn_export['summary'].astype(str)

    upload_dataframe(
        client, CLEAN_BUCKET,
        'ghcn_daily_cleaned.parquet',
        ghcn_export,
        file_format='parquet',
    )
    print(f'Uploaded {len(ghcn_export):,} rows to {CLEAN_BUCKET}/ghcn_daily_cleaned.parquet')

if len(meteo_all) > 0:
    # Open-Meteo: select relevant columns
    meteo_cols = ['date', 'location', 'source']
    if 'temperature_2m_max' in meteo_all.columns:
        meteo_cols.append('temperature_2m_max')
    if 'temperature_2m_min' in meteo_all.columns:
        meteo_cols.append('temperature_2m_min')
    if 'temperature_2m_mean' in meteo_all.columns:
        meteo_cols.append('temperature_2m_mean')
    if 'precipitation_sum' in meteo_all.columns:
        meteo_cols.append('precipitation_sum')
    if 'summary' in meteo_all.columns:
        meteo_cols.append('summary')

    meteo_export = meteo_all[meteo_cols].copy()
    meteo_export['summary'] = meteo_export['summary'].astype(str)

    upload_dataframe(
        client, CLEAN_BUCKET,
        'open_meteo_daily_cleaned.parquet',
        meteo_export,
        file_format='parquet',
    )
    print(f'Uploaded {len(meteo_export):,} rows to {CLEAN_BUCKET}/open_meteo_daily_cleaned.parquet')

In [ ]:
# Cell 22 — Load cleaned data into DuckDB for cross-source queries
# This populates the weather_observations_raw table that Notebook 02
# uses for comparing app forecasts to real weather.

from minio_helper import ensure_bucket

ANALYTICS_BUCKET = 'weather-analytics'
DUCKDB_OBJECT = 'duckdb/weather.duckdb'
DUCKDB_LOCAL = '/tmp/weather_nb03.duckdb'

# Download existing DuckDB if it exists (CDC DAG may have created it)
ensure_bucket(client, ANALYTICS_BUCKET)
if object_exists(client, ANALYTICS_BUCKET, DUCKDB_OBJECT):
    client.fget_object(ANALYTICS_BUCKET, DUCKDB_OBJECT, DUCKDB_LOCAL)
    print('Downloaded existing DuckDB from MinIO')
else:
    print('No existing DuckDB — will create a new one')

con = duckdb.connect(DUCKDB_LOCAL)

# Create the observations table
con.execute("""
    CREATE TABLE IF NOT EXISTS weather_observations_clean (
        date           DATE,
        location       VARCHAR,
        source         VARCHAR,
        temp_max_c     DOUBLE,
        temp_min_c     DOUBLE,
        temp_mean_c    DOUBLE,
        precipitation_mm DOUBLE,
        summary        VARCHAR,
        loaded_at      TIMESTAMP
    )
""")

# Clear existing data and reload (idempotent)
con.execute("DELETE FROM weather_observations_clean")

# Insert GHCN data
if len(ghcn_all) > 0:
    ghcn_for_db = ghcn_export.copy()
    ghcn_for_db = ghcn_for_db.drop(columns=['station_id'], errors='ignore')
    con.execute("INSERT INTO weather_observations_clean SELECT *, CURRENT_TIMESTAMP FROM ghcn_for_db")
    print(f'Inserted {len(ghcn_for_db):,} GHCN rows into DuckDB')

# Insert Open-Meteo data (need to align column names)
if len(meteo_all) > 0:
    meteo_for_db = pd.DataFrame({
        'date': meteo_export['date'],
        'location': meteo_export['location'],
        'source': meteo_export['source'],
        'temp_max_c': meteo_export.get('temperature_2m_max'),
        'temp_min_c': meteo_export.get('temperature_2m_min'),
        'temp_mean_c': meteo_export.get('temperature_2m_mean'),
        'precipitation_mm': meteo_export.get('precipitation_sum'),
        'summary': meteo_export['summary'],
    })
    con.execute("INSERT INTO weather_observations_clean SELECT *, CURRENT_TIMESTAMP FROM meteo_for_db")
    print(f'Inserted {len(meteo_for_db):,} Open-Meteo rows into DuckDB')

# Verify
count = con.execute("SELECT COUNT(*) FROM weather_observations_clean").fetchone()[0]
print(f'\nTotal rows in weather_observations_clean: {count:,}')

con.close()

# Upload updated DuckDB back to MinIO
from minio_helper import upload_file
upload_file(client, ANALYTICS_BUCKET, DUCKDB_OBJECT, DUCKDB_LOCAL)
print(f'Uploaded DuckDB to MinIO: {ANALYTICS_BUCKET}/{DUCKDB_OBJECT}')

# Clean up
os.remove(DUCKDB_LOCAL)
print('Done!')

## Summary

In this notebook we:

1. **Loaded** raw GHCN-Daily data from MinIO (long format, tenths of degrees)
2. **Pivoted** from long to wide format using `pivot_table()`
3. **Handled missing values** with interpolation (temperatures) and zero-fill (precipitation)
4. **Converted units** from tenths of degrees to actual degrees Celsius
5. **Mapped temperatures** to our app's 10 Summary labels using `pd.cut()`
6. **Processed all stations** and both data sources (GHCN + Open-Meteo)
7. **Saved cleaned data** to MinIO as Parquet and loaded into DuckDB

### Next Steps

In **Notebook 04**, we'll use this cleaned data to build **statistical weather profiles**
that can replace the random number generator in our minion scheduler.